In [ ]:
import numpy as np
import pandas as pd
import rasterio
import os
import glob

import shap
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier



# ==================================================
# 1.路径
# ==================================================

folder = str(config.FEATURES)
label_file = str(config.SAMPLE_PUL)
out_dir = str(config.OUTPUT / "pu_ca_rf_shap")

os.makedirs(out_dir, exist_ok=True)



# ==================================================
# 2.读取RF和CA TIF
# ==================================================

rf_files = sorted(
    glob.glob(folder + r"\*_RF.tif")
)

ca_files = sorted(
    glob.glob(folder + r"\*_CA.tif")
)


all_files = rf_files + ca_files


print("RF:", len(rf_files),
      "CA:", len(ca_files))


print("Total variables:",
      len(all_files))


# 检查

if len(all_files) != 22:
    raise Exception(
        "输入变量不是22个，请检查tif文件"
    )

# =========================
# 2. reference grid（统一栅格）
# =========================
ref_path = rf_files[0]

with rasterio.open(ref_path) as ref:
    ref_shape = (ref.height, ref.width)
    profile = ref.profile.copy()
    h, w = ref_shape

# =========================
# 3. 栅格对齐函数（不依赖CRS）
# =========================
def align_no_crs(f):

    with rasterio.open(f) as src:
        data = src.read(1).astype(np.float32)

        out = np.full(ref_shape, np.nan, dtype=np.float32)

        hh = min(h, data.shape[0])
        ww = min(w, data.shape[1])

        out[:hh, :ww] = data[:hh, :ww]

    return out

# ==================================================
# 3.变量名称
# ==================================================

feature_names=[]


for f in rf_files:

    name=os.path.basename(f)

    name=name.replace(
        ".tif",
        ""
    )

    feature_names.append(name)



for f in ca_files:

    name=os.path.basename(f)

    name=name.replace(
        ".tif",
        ""
    )

    feature_names.append(name)



print(feature_names)


# ==================================================
# 4.读取所有TIF并统一尺寸
# ==================================================

layers=[]


for f in all_files:

    data = align_no_crs(f)

    layers.append(data)

    print(
        os.path.basename(f),
        data.shape
    )


layers=np.stack(layers)


print(
    "Raster shape:",
    layers.shape
)
# ==================================================
# 5.转换成RF输入格式
# ==================================================
X = layers.reshape(
        len(all_files),
        -1
).T


print(
    "X:",
    X.shape
)



print(
    "X:",
    X.shape
)



# ==================================================
# 6.去除NaN
# ==================================================

mask=np.all(
    np.isfinite(X),
    axis=1
)


X_clean=X[mask]



# ==================================================
# 7.读取PU训练样本坐标
# ==================================================

sample = pd.read_csv(
    label_file,
    sep=",",
    header=None
)


print(sample.head())

print(
    "样本数量:",
    sample.shape
)


# 第1列 X坐标
# 第2列 Y坐标
# 第3列 标签

sample_x = sample.iloc[:,0].values

sample_y = sample.iloc[:,1].values

sample_label = sample.iloc[:,2].values



print(
    "标签统计:",
    np.unique(
        sample_label,
        return_counts=True
    )
)
# ==================================================
# 8.根据坐标匹配TIF像元
# ==================================================

from rasterio.transform import rowcol


# 参考栅格transform

with rasterio.open(ref_path) as src:

    transform = src.transform



rows_sample, cols_sample = rowcol(

    transform,

    sample_x,

    sample_y

)


rows_sample = np.array(rows_sample)

cols_sample = np.array(cols_sample)



# 去除超出范围点

valid = (

    (rows_sample >=0) &
    (rows_sample < h) &
    (cols_sample >=0) &
    (cols_sample < w)

)



rows_sample = rows_sample[valid]

cols_sample = cols_sample[valid]

sample_label = sample_label[valid]



print(
    "有效训练点:",
    len(sample_label)
)



# ==================================================
# 9.提取训练样本对应的22个变量
# ==================================================


sample_index = (

    rows_sample * w

    +

    cols_sample

)



# 从全部像元X中提取

X_shap = X[sample_index]


y_shap = sample_label



print(
"X_shap:",
X_shap.shape
)


print(
"y_shap:",
y_shap.shape
)



# ==================================================
# 10.训练PU-CA-RF
# ==================================================


rf_shap = RandomForestClassifier(

    n_estimators=300,

    max_depth=None,

    random_state=42,

    n_jobs=-1

)



rf_shap.fit(

    X_shap,

    y_shap

)



print(
"SHAP RF训练完成"
)



# ==================================================
# 11.SHAP计算
# ==================================================

explainer = shap.TreeExplainer(

    rf_shap

)


shap_values = explainer(

    X_shap

)



print(
"SHAP计算完成"
)



# ==================================================
# 12.处理SHAP维度
# ==================================================

sv = shap_values.values



print(
"SHAP原始维度:",
sv.shape
)



if sv.ndim == 3:

    sv = sv[:,:,1]



print(
"修正后SHAP维度:",
sv.shape
)



# ==================================================
# 13.SHAP summary图
# ==================================================


shap.summary_plot(

    sv,

    X_shap,

    feature_names=feature_names

)


plt.savefig(

    os.path.join(

        out_dir,

        "SHAP_summary.png"

    ),

    dpi=300,

    bbox_inches="tight"

)



plt.close()



# ==================================================
# 14.SHAP变量重要性
# ==================================================


shap.summary_plot(

    sv,

    X_shap,

    feature_names=feature_names,

    plot_type="bar"

)



plt.savefig(

    os.path.join(

        out_dir,

        "SHAP_bar.png"

    ),

    dpi=300,

    bbox_inches="tight"

)



plt.close()



# ==================================================
# 15.单变量SHAP影响
# ==================================================


pb_index = feature_names.index(

    "Pb_RF"

)



shap.dependence_plot(

    pb_index,

    sv,

    X_shap,

    feature_names=feature_names

)



plt.savefig(

    os.path.join(

        out_dir,

        "Pb_RF_dependence.png"

    ),

    dpi=300,

    bbox_inches="tight"

)



plt.close()



# ==================================================
# 16.导出SHAP表
# ==================================================


df_shap = pd.DataFrame(

    sv,

    columns=feature_names

)



df_shap.to_csv(

    os.path.join(

        out_dir,

        "SHAP_all_variables.csv"

    ),

    index=False

)



print(
"SHAP CSV保存完成"
)



# ==================================================
# 17.SHAP空间恢复
# ==================================================


full_shap = np.full(

    (

        X.shape[0],

        len(feature_names)

    ),

    np.nan

)



full_shap[sample_index] = sv



full_shap = full_shap.reshape(

    h,

    w,

    len(feature_names)

)



# ==================================================
# 18.输出SHAP空间TIF
# ==================================================


for i,name in enumerate(feature_names):


    out_tif = os.path.join(

        out_dir,

        name+"_SHAP.tif"

    )


    profile.update(

        dtype="float32",

        count=1,

        nodata=np.nan

    )


    with rasterio.open(

        out_tif,

        "w",

        **profile

    ) as dst:


        dst.write(

            full_shap[:,:,i]
            .astype("float32"),

            1

        )



    print(
        name,
        "完成"
    )



print("======================")
print("PU-CA-RF SHAP全部完成")
print("======================")